# ETL Pipeline
In this file, we read the raw dataset and perform data cleaning and preprocessing before writing the processed dataset into our dataset/processed directory. This processed dataset will be used for all future analysis.

#### Import Libraries and Initialize Spark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, regexp_extract, unix_timestamp, lit, when, lower, coalesce, to_date
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

import kagglehub
import os
import shutil


# Initialize Spark session
spark = (
    SparkSession.builder
        .appName("LendingClubPreprocessing")
        .config("spark.driver.memory", "8g")
        .config("spark.executor.memory", "16g")
        .config("spark.sql.shuffle.partitions", "200")
        .config("spark.driver.maxResultSize", "4g")
        .getOrCreate()
)


#### Clean the numeric string columns term and employment length

In [ ]:
def clean_string_columns(df):
    # emp_length cleaning: we will convert: "< 1 year" → 0, "10+ years" → 10, "n/a" or invalid → null, "X years" → X
    df = df.withColumn(
            "emp_length",
            when(lower(col("emp_length")).rlike("n/a"), lit(None))  # n/a → null
            .when(lower(col("emp_length")).rlike("< 1"), lit(0))    # <1 → 0
            .when(lower(col("emp_length")).rlike("10"), lit(10))    # 10+ → 10
            .when(lower(col("emp_length")).rlike("year"), 
                  regexp_extract("emp_length", r"(\d+)", 0).cast("int"))
            .otherwise(lit(None))                                   # invalid → null
        )

    # term: convert "36 months" → 36
    df = df.withColumn(
        'term',
        regexp_extract('term', r'(\d+)', 0).cast('int')
    )

    return df

#### Unify Fico Scores
Since the dataset provides the upper and lower boundaries of the borrower's fico score, we can average them to find a unified fico score.

In [ ]:
def unify_fico_score(df):
    fico_cols = [col for col in df.columns if 'fico' in col.lower()]

    if fico_cols:
        if 'fico_range_low' in df.columns and 'fico_range_high' in df.columns:
            df = df.withColumn('fico_score', (col('fico_range_low') + col('fico_range_high')) / 2)
        elif len(fico_cols) > 0:
            df = df.withColumn('fico_score', col(fico_cols[0]))
    else:
        df = df.withColumn('fico_score', lit(700))  # Default FICO score

    return df

#### Parse date columns
Since our dataset has several date columns, we need to make them consistent with Spark's DateType.

In [ ]:
def parse_date_columns(df):
    date_cols = ['issue_d', 'earliestCrLine', 'acceptD', 'creditPullD', 'listD', 'expD']

    for col_name in date_cols:
        if col_name in df.columns:
            df = df.withColumn(col_name, 
                            coalesce(
                                to_date(col(col_name), 'MMM-yyyy'),
                                to_date(col(col_name), 'yyyy-MM-dd')
                            )
            )
    return df

#### Handle missing values
For numeric columns, we chose to replace missing values with the median of that column as it's less sensitive to outliers than the mean and less likely to harm our analysis.
For categorical columns, we chose to replace missing values with the mode of that column as it's the most probable value.

In [ ]:
def handle_missing_values(df):
    # For numeric columns: we take the median
    numeric_columns = [c for c, t in df.dtypes if t in ('int', 'double', 'bigint', 'float')]
    
    if numeric_columns:
        medians = df.approxQuantile(numeric_columns, [0.5], 0)
        median_dict = {col_name: medians[i][0] for i, col_name in enumerate(numeric_columns)}
        df = df.fillna(median_dict)
    
    print("Filled numeric missing values with medians.")
    
    # For categorical columns: we take the mode
    categorical_columns = [c for c, t in df.dtypes if t == 'string']
    for col_name in categorical_columns:
        w = Window.partitionBy(col_name)
        df = df.withColumn("cnt", F.count("*").over(w))
        mode_row = df.select(col_name, "cnt").orderBy(F.desc("cnt")).limit(1).collect()
        if mode_row and mode_row[0][0] is not None:
            mode_value = mode_row[0][0]
            df = df.fillna({col_name: mode_value})
        df = df.drop("cnt")
    
    return df

#### Remove Duplicate Columns
In case we have duplicate columns, we will keep the first occurring one and drop the rest. Printing the dropped columns for future potential debugging.

In [ ]:
def remove_duplicate_columns(df):
    cols = df.columns
    seen = set()
    duplicates = []

    for c in cols:
        if c in seen:
            duplicates.append(c)
        else:
            seen.add(c)
    
    if duplicates:
        print(f"Dropping duplicate columns: {duplicates}")
        df = df.drop(*duplicates)  # drop all duplicates at once
    else:
        print("No duplicate columns found.")

    return df

#### Read the raw dataset

In [ ]:
# Defining the schema for the Lending Club dataset
lending_schema = StructType([
    StructField("id", StringType(), True),
    StructField("member_id", StringType(), True),
    StructField("loan_amnt", IntegerType(), True),
    StructField("funded_amnt", IntegerType(), True),
    StructField("funded_amnt_inv", DoubleType(), True),
    StructField("term", StringType(), True),
    StructField("int_rate", DoubleType(), True),
    StructField("installment", DoubleType(), True),
    StructField("grade", StringType(), True),
    StructField("sub_grade", StringType(), True),
    StructField("emp_title", StringType(), True),
    StructField("emp_length", StringType(), True),
    StructField("home_ownership", StringType(), True),
    StructField("annual_inc", StringType(), True),
    StructField("verification_status", StringType(), True),
    StructField("issue_d", StringType(), True),
    StructField("loan_status", StringType(), True),
    StructField("pymnt_plan", StringType(), True),
    StructField("url", StringType(), True),
    StructField("desc", StringType(), True),
    StructField("purpose", StringType(), True),
    StructField("title", StringType(), True),
    StructField("zip_code", StringType(), True),
    StructField("addr_state", StringType(), True),
    StructField("dti", StringType(), True),
    StructField("delinq_2yrs", StringType(), True),
    StructField("earliest_cr_line", StringType(), True),
    StructField("inq_last_6mths", StringType(), True),
    StructField("mths_since_last_delinq", StringType(), True),
    StructField("mths_since_last_record", StringType(), True),
    StructField("open_acc", StringType(), True),
    StructField("pub_rec", StringType(), True),
    StructField("revol_bal", StringType(), True),
    StructField("revol_util", StringType(), True),
    StructField("total_acc", StringType(), True),
    StructField("initial_list_status", StringType(), True),
    StructField("out_prncp", StringType(), True),
    StructField("out_prncp_inv", StringType(), True),
    StructField("total_pymnt", StringType(), True),
    StructField("total_pymnt_inv", StringType(), True),
    StructField("total_rec_prncp", StringType(), True),
    StructField("total_rec_int", StringType(), True),
    StructField("total_rec_late_fee", StringType(), True),
    StructField("recoveries", StringType(), True),
    StructField("collection_recovery_fee", StringType(), True),
    StructField("last_pymnt_d", StringType(), True),
    StructField("last_pymnt_amnt", StringType(), True),
    StructField("next_pymnt_d", StringType(), True),
    StructField("last_credit_pull_d", StringType(), True),
    StructField("collections_12_mths_ex_med", StringType(), True),
    StructField("mths_since_last_major_derog", StringType(), True),
    StructField("policy_code", StringType(), True),
    StructField("application_type", StringType(), True),
    StructField("annual_inc_joint", StringType(), True),
    StructField("dti_joint", StringType(), True),
    StructField("verification_status_joint", StringType(), True),
    StructField("acc_now_delinq", StringType(), True),
    StructField("tot_coll_amt", StringType(), True),
    StructField("tot_cur_bal", StringType(), True),
    StructField("open_acc_6m", StringType(), True),
    StructField("open_act_il", StringType(), True),
    StructField("open_il_12m", StringType(), True),
    StructField("open_il_24m", StringType(), True),
    StructField("mths_since_rcnt_il", StringType(), True),
    StructField("total_bal_il", StringType(), True),
    StructField("il_util", StringType(), True),
    StructField("open_rv_12m", StringType(), True),
    StructField("open_rv_24m", StringType(), True),
    StructField("max_bal_bc", StringType(), True),
    StructField("all_util", StringType(), True),
    StructField("total_rev_hi_lim", StringType(), True),
    StructField("inq_fi", StringType(), True),
    StructField("total_cu_tl", StringType(), True),
    StructField("inq_last_12m", StringType(), True),
    StructField("acc_open_past_24mths", IntegerType(), True),
    StructField("avg_cur_bal", StringType(), True),
    StructField("bc_open_to_buy", IntegerType(), True),
    StructField("bc_util", StringType(), True),
    StructField("chargeoff_within_12_mths", DoubleType(), True),
    StructField("delinq_amnt", IntegerType(), True),
    StructField("mo_sin_old_il_acct", DoubleType(), True),
    StructField("mo_sin_old_rev_tl_op", IntegerType(), True),
    StructField("mo_sin_rcnt_rev_tl_op", IntegerType(), True),
    StructField("mo_sin_rcnt_tl", IntegerType(), True),
    StructField("mort_acc", IntegerType(), True),
    StructField("mths_since_recent_bc", IntegerType(), True),
    StructField("mths_since_recent_bc_dlq", IntegerType(), True),
    StructField("mths_since_recent_inq", IntegerType(), True),
    StructField("mths_since_recent_revol_delinq", IntegerType(), True),
    StructField("num_accts_ever_120_pd", IntegerType(), True),
    StructField("num_actv_bc_tl", IntegerType(), True),
    StructField("num_actv_rev_tl", IntegerType(), True),
    StructField("num_bc_sats", IntegerType(), True),
    StructField("num_bc_tl", IntegerType(), True),
    StructField("num_il_tl", IntegerType(), True),
    StructField("num_op_rev_tl", IntegerType(), True),
    StructField("num_rev_accts", IntegerType(), True),
    StructField("num_rev_tl_bal_gt_0", IntegerType(), True),
    StructField("num_sats", IntegerType(), True),
    StructField("num_tl_120dpd_2m", IntegerType(), True),
    StructField("num_tl_30dpd", IntegerType(), True),
    StructField("num_tl_90g_dpd_24m", IntegerType(), True),
    StructField("num_tl_op_past_12m", IntegerType(), True),
    StructField("pct_tl_nvr_dlq", DoubleType(), True),
    StructField("percent_bc_gt_75", DoubleType(), True),
    StructField("pub_rec_bankruptcies", IntegerType(), True),
    StructField("tax_liens", IntegerType(), True),
    StructField("tot_hi_cred_lim", IntegerType(), True),
    StructField("total_bal_ex_mort", IntegerType(), True),
    StructField("total_bc_limit", IntegerType(), True),
    StructField("total_il_high_credit_limit", IntegerType(), True),
    StructField("revol_bal_joint", IntegerType(), True),
    StructField("sec_app_earliest_cr_line", StringType(), True),
    StructField("sec_app_inq_last_6mths", IntegerType(), True),
    StructField("sec_app_mort_acc", IntegerType(), True),
    StructField("sec_app_open_acc", IntegerType(), True),
    StructField("sec_app_revol_util", DoubleType(), True),
    StructField("sec_app_open_act_il", IntegerType(), True),
    StructField("sec_app_num_rev_accts", IntegerType(), True),
    StructField("sec_app_chargeoff_within_12_mths", IntegerType(), True),
    StructField("sec_app_collections_12_mths_ex_med", IntegerType(), True),
    StructField("sec_app_mths_since_last_major_derog", IntegerType(), True),
    StructField("hardship_flag", StringType(), True),
    StructField("hardship_type", StringType(), True),
    StructField("hardship_reason", StringType(), True),
    StructField("hardship_status", StringType(), True),
    StructField("deferral_term", StringType(), True),
    StructField("hardship_amount", StringType(), True),
    StructField("hardship_start_date", StringType(), True),
    StructField("hardship_end_date", StringType(), True),
    StructField("payment_plan_start_date", StringType(), True),
    StructField("hardship_length", StringType(), True),
    StructField("hardship_dpd", StringType(), True),
    StructField("hardship_loan_status", StringType(), True),
    StructField("orig_projected_additional_accrued_interest", StringType(), True),
    StructField("hardship_payoff_balance_amount", StringType(), True),
    StructField("hardship_last_payment_amount", StringType(), True),
    StructField("disbursement_method", StringType(), True),
    StructField("debt_settlement_flag", StringType(), True),
    StructField("debt_settlement_flag_date", StringType(), True),
    StructField("settlement_status", StringType(), True),
    StructField("settlement_date", StringType(), True),
    StructField("settlement_amount", StringType(), True),
    StructField("settlement_percentage", StringType(), True),
    StructField("settlement_term", StringType(), True)
])

In [ ]:

path = kagglehub.dataset_download("adarshsng/lending-club-loan-data-csv")
print("KaggleHub path:", path)

SRC_PATH = os.path.join(path, "loan.csv")

DEST_PATH = "../dataset/raw/loan.csv"

os.makedirs("../dataset/raw", exist_ok=True)

shutil.copy(SRC_PATH, DEST_PATH)

print("Dataset copied to:", DEST_PATH)

df = spark.read.csv(DEST_PATH, header=True, schema=lending_schema)
print(f"Loaded dataset with {df.count()} rows and {len(df.columns)} columns.")
df.show(5)

# Testing mode
df = df.limit(100000)

#### Call all the ETL functions on the dataframe

In [ ]:
df = clean_string_columns(df)
print("Cleaned string columns.")

In [ ]:
df = unify_fico_score(df)
print("Unified FICO score.")

In [ ]:
df = parse_date_columns(df)
print("Date columns parsed to standard format.")

In [ ]:
df = handle_missing_values(df)
print("Handled missing values.")

In [ ]:
df = remove_duplicate_columns(df)
print("Removed duplicate columns if any.")

#### Save the processed data

In [ ]:
import os
PARQUET_PATH = os.path.abspath("../dataset/processed")
df.write.mode("overwrite").parquet(PARQUET_PATH)
print(f"Preprocessed data saved to {PARQUET_PATH}")

The below cell may be useful for when we shift to year-based partitioning.

In [ ]:
# import os
# from pyspark.sql.utils import AnalysisException

# def preprocess_by_year(df, categorical_columns, target_column):
#     for cat_feature in categorical_columns:
#         encoded_column_name = f'{cat_feature}_encoded'
#         if encoded_column_name in df.columns:
#             print(f"Dropping {encoded_column_name} column.") 
#             df = df.drop(encoded_column_name)

#     df = clean_string_columns(df)   
#     df = unify_fico_score(df)       
#     df = handle_missing_values(df)  
#     df = feature_engineering(df)    

#     df, encoded_columns = encode_categorical_columns(df, categorical_columns)

#     df = finalize_features(df, categorical_columns, encoded_columns)

#     df = remove_duplicate_columns(df)

#     return df

# categorical_columns = ['home_ownership', 'purpose']
# target_column = 'loan_status_binary'

# df = spark.read.parquet("../../accepted_by_year") 

# years = df.select("year").distinct().rdd.flatMap(lambda x: x).collect()

# for year in years:
#     df_year = df.filter(df["year"] == year)
    
#     print(f"Columns in raw data for year {year}: {df_year.columns}")

#     df_year_preprocessed = preprocess_by_year(df_year, categorical_columns, target_column)
    
#     print(f"Columns before saving for year {year}: {df_year_preprocessed.columns}")

#     output_path = f'../../preprocessed/year={year}'
#     if not os.path.exists(output_path):
#         os.makedirs(output_path) 
    
#     try:
#         df_year_preprocessed.write.mode("overwrite").parquet(output_path)
#         print(f"Preprocessed data for year {year} saved.")
#     except AnalysisException as e:
#         print(f"Error occurred while saving data for year {year}: {e}")